# Run vanilla and fine-tuned Laya on the test sets

This notebook loads the two checkpoints shipped in `checkpoints/` and asks both of them every question in the hand-labelled test sets:

- **Search routing:** 96 catalog queries. Does each one need the graph search? The spatial search?
- **First tool:** 42 held-out analysis requests. Which of 53 tools should run first?

It then shows the accuracy, every individual answer, and where the two models disagree. `laya_decision_models.ipynb` explains what a decision model is; this notebook only runs the models.

**Runtime:** about 1 minute on a GPU. On a 2-CPU machine it takes about 20 minutes, almost all of it the 53-option tool questions. Set `TOOL_LIMIT` below to run fewer.

In [1]:
import json, time, warnings
warnings.filterwarnings("ignore", message="IProgress not found")
from pathlib import Path

import pandas as pd
import torch
from laya.agent import Agent

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 70)

CHECKPOINTS = {"vanilla": Path("checkpoints/laya-vanilla"), "fine-tuned": Path("checkpoints/laya-finetuned")}
for d in CHECKPOINTS.values():
    w = d / "model.safetensors"
    assert w.exists() and w.stat().st_size > 10_000_000, f"{w} is missing or a Git LFS pointer: run `git lfs pull`"

QUESTIONS = json.loads(Path("data/questions.json").read_text(encoding="utf-8"))
# Each model is asked the router questions in the wording that suits it: the fine-tuned model the exact wording
# it was trained on, vanilla Laya the wording that scored best for it zero-shot.
ROUTER_WORDING = {"vanilla": QUESTIONS["router_zero_shot"], "fine-tuned": QUESTIONS["router_finetuned"]}
TOOL_Q = {"tool": QUESTIONS["tool"]}

read = lambda p: [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
ROUTER_SET = read("data/eval/router_eval_queries.jsonl")
TOOL_SET = read("data/eval/tool_eval_holdout.jsonl")
TOOL_LIMIT = None          # e.g. 10 for a quick run on CPU

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE} · {len(ROUTER_SET)} routing queries · {len(TOOL_SET[:TOOL_LIMIT])} tool requests")
print("\nrouter questions, as each model is asked them:")
for name, q in ROUTER_WORDING.items():
    print(f"  {name:10} graph:   {q['graph']['instructions']}\n  {'':10} spatial: {q['spatial']['instructions']}")

device: cuda · 96 routing queries · 42 tool requests

router questions, as each model is asked them:
  vanilla    graph:   Does this query name a specific person, organization or tag?
             spatial: Does this query mention a place or geographic location?
  fine-tuned graph:   Does this query name a specific person, organization or tag, ask what is popular, or ask for resources linked to another resource?
             spatial: Does this query mention a specific place, coordinates, or a distance or area on a map?


## Call the models

One model at a time is loaded, so peak memory stays at about 2.5 GB on CPU. Each routing query is one forward pass that answers both yes/no questions. Each tool request is one forward pass over all 53 options.

In [2]:
def run(name):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = Agent(str(CHECKPOINTS[name]), device=DEVICE)
    t0 = time.perf_counter()
    router = model.predict_batch([r["query"] for r in ROUTER_SET], ROUTER_WORDING[name], batch_size=16)
    t1 = time.perf_counter()
    tools = model.predict_batch([f"{r['context']}\nRequest: {r['request']}" for r in TOOL_SET[:TOOL_LIMIT]], TOOL_Q,
                                batch_size=4 if DEVICE == "cuda" else 1, max_len=1600, head_max_len=1400)
    t2 = time.perf_counter()
    print(f"{name:10} routing: {(t1 - t0) / len(ROUTER_SET) * 1000:6.0f} ms/query · "
          f"tools: {(t2 - t1) / len(tools) * 1000:6.0f} ms/request")
    return {"router": [{f: a["answers"][f]["noul"] for f in ("graph", "spatial")} for a in router],
            "tools": [a["answers"]["tool"]["probabilities"] for a in tools]}


results = {}
for name in CHECKPOINTS:
    results[name] = run(name)

vanilla    routing:     17 ms/query · tools:    138 ms/request


fine-tuned routing:     12 ms/query · tools:    135 ms/request


## Search routing: accuracy

Scored on the 87 queries not marked ambiguous, with a cut-off of 0.5.

In [3]:
clear = [i for i, r in enumerate(ROUTER_SET) if not r.get("ambiguous")]
acc = {}
for name, res in results.items():
    pred = [{f: res["router"][i][f] >= 0.5 for f in ("graph", "spatial")} for i in range(len(ROUTER_SET))]
    acc[name] = {"graph accuracy": sum(pred[i]["graph"] == ROUTER_SET[i]["graph"] for i in clear) / len(clear),
                 "spatial accuracy": sum(pred[i]["spatial"] == ROUTER_SET[i]["spatial"] for i in clear) / len(clear),
                 "both right": sum(pred[i]["graph"] == ROUTER_SET[i]["graph"] and pred[i]["spatial"] == ROUTER_SET[i]["spatial"]
                                   for i in clear) / len(clear)}
pd.DataFrame(acc).T.round(3)

,graph accuracy,spatial accuracy,both right
vanilla,0.621,0.667,0.414
fine-tuned,0.954,0.989,0.943


## Search routing: every answer

Each cell shows P(yes); ✗ marks a disagreement with the gold label. Rows marked *amb.* are ambiguous and not scored.

In [4]:
def cell(p, gold):
    return f"{p:.2f}" + ("" if (p >= 0.5) == gold else " ✗")


rows = []
for i, r in enumerate(ROUTER_SET):
    row = {"query": r["query"][:70] + (" (amb.)" if r.get("ambiguous") else "")}
    for f in ("graph", "spatial"):
        row[(f, "gold")] = "yes" if r[f] else "no"
        for name in results:
            row[(f, name)] = cell(results[name]["router"][i][f], r[f])
    rows.append(row)
router_table = pd.DataFrame(rows).set_index("query")
router_table.columns = pd.MultiIndex.from_tuples(router_table.columns)
router_table

graph  \
                                                                        gold   
query                                                                          
Which Chicago community areas have the poorest spatial accessibility t    no   
Use the available flood-depth prediction resources to identify require    no   
Find flood risk maps for Boone County, Iowa, compare available map yea    no   
Build an OSM-based road network for a wildfire evacuation scenario and    no   
Droughts in California                                                    no   
Floods in Texas                                                           no   
datasets about flooding in Illinois                                       no   
water datasets                                                            no   
Water quality datasets in California                                      no   
What are the main sources of greenhouse gas emissions?                    no   
What datasets exist for floods?                                           no   
climate geospatial data                                                   no   
flood risk geospatial datasets                                            no   
geospatial datasets dams                                                  no   
water resource management irrigation                                      no   
publications by John Smith                                               yes   
datasets from NASA                                                       yes   
resources tagged climate                                                 yes   
papers affiliated with USGS                                              yes   
notebooks written by Shaowen Wang                                        yes   
What has Anand Padmanabhan published on I-GUIDE?                         yes   
most popular notebooks                                                   yes   
trending datasets this month                                             yes   
resources developed by the University of Illinois                        yes   
datasets maintained by NOAA                                              yes   
notebooks that use the SVI dataset                                       yes   
what publications cite the CyberGIS-Compute paper?                       yes   
show me resources related to element id 3f2a9c1e                         yes   
who contributed the FireABM notebook?                                    yes   
which organizations have uploaded the most datasets?                     yes   
notebooks categorized as hydrology                                       yes   
educational resources labeled machine learning                           yes   
other datasets by the author of the SNAP dataset                         yes   
notebooks connected to the Pysal Access publication                      yes   
top viewed OERs                                                          yes   
weather stations near Chicago                                             no   
land cover data within 50 miles of Denver                                 no   
datasets covering 40.1N, 88.2W                                            no   
elevation data for the Grand Canyon                                       no   
heat vulnerability in Phoenix neighborhoods                               no   
crop yield data for the Midwest                                           no   
hurricane damage along the Gulf Coast                                     no   
air quality monitoring in Los Angeles County                              no   
groundwater depletion in the Ogallala aquifer                             no   
Sentinel-2 imagery over Nairobi                                           no   
census tracts in Champaign County                                         no   
wildfire perimeters in northern California 2020                           no   
flood maps for the Mississippi River basin                             

## First tool: accuracy

A request counts as right when the top choice is one of the tools the gold set accepts.

In [5]:
tool_rows = TOOL_SET[:TOOL_LIMIT]
choice = {name: [max(p, key=p.get) for p in res["tools"]] for name, res in results.items()}
pd.DataFrame({name: {"accuracy": sum(c in r["gold"] for c, r in zip(choice[name], tool_rows)) / len(tool_rows),
                     "requests": len(tool_rows)} for name in results}).T.round(3).astype({"requests": int})

,accuracy,requests
vanilla,0.095,42
fine-tuned,0.786,42


## First tool: every answer

The top choice and its probability; ✓ means it is among the gold tools.

In [6]:
def top(probs, gold):
    best, p = max(probs.items(), key=lambda kv: kv[1])
    return f"{'✓' if best in gold else '✗'} {best} {p:.2f}"


pd.DataFrame([{"request": r["request"][:65],
               "files": "yes" if r["context"] != "Attached files: none" else "no",
               "gold": ", ".join(r["gold"]),
               **{name: top(results[name]["tools"][i], r["gold"]) for name in results}}
              for i, r in enumerate(tool_rows)]).set_index("request")

,files,gold,vanilla,fine-tuned
request,,,,
"Pull the boundary of Travis County, Texas.",no,admin_boundary,✗ add_map_layer 0.48,✓ admin_boundary 1.00
"Outline Madison, Wisconsin on the map.",no,admin_boundary,✗ add_map_layer 0.68,✓ admin_boundary 1.00
I need the tracts inside Harris County.,no,admin_boundary,✗ add_map_layer 0.57,✓ admin_boundary 1.00
Show the state line of Nevada.,no,admin_boundary,✗ add_map_layer 0.60,✓ admin_boundary 1.00
Where is the Space Needle? Pin it on the map.,no,geocode_places,✗ add_map_layer 0.68,✗ none 0.20
Get lat/lon for Argonne National Laboratory and Fermilab.,no,geocode_places,✗ add_map_layer 0.59,✓ geocode_places 0.17
"Plot the locations of Stanford, MIT and Caltech.",no,geocode_places,✗ add_map_layer 0.57,✓ geocode_places 0.59
Write and run code that fits a linear regression to 50 random poi,no,execute_code,✗ add_map_layer 0.62,✓ execute_code 0.66
Use Python to convert 12 miles to kilometres and print it.,no,execute_code,✗ add_map_layer 0.55,✓ execute_code 0.64


## Where the two models disagree

Routing queries where vanilla and fine-tuned Laya reach a different decision on either question, and who matches the gold label.

In [7]:
diff = []
for i, r in enumerate(ROUTER_SET):
    v, f = results["vanilla"]["router"][i], results["fine-tuned"]["router"][i]
    for q in ("graph", "spatial"):
        if (v[q] >= 0.5) != (f[q] >= 0.5):
            diff.append({"query": r["query"][:70], "question": q, "gold": "yes" if r[q] else "no",
                         "vanilla": cell(v[q], r[q]), "fine-tuned": cell(f[q], r[q])})
print(f"{len(diff)} decisions differ out of {2 * len(ROUTER_SET)}")
pd.DataFrame(diff).set_index(["query", "question"])

64 decisions differ out of 192


,,gold,vanilla,fine-tuned
query,question,,,
Use the available flood-depth prediction resources to identify require,graph,no,0.03,0.80 ✗
"Find flood risk maps for Boone County, Iowa, compare available map yea",spatial,yes,0.18 ✗,1.00
Floods in Texas,spatial,yes,0.31 ✗,1.00
datasets about flooding in Illinois,spatial,yes,0.20 ✗,1.00
Water quality datasets in California,spatial,yes,0.16 ✗,1.00
publications by John Smith,graph,yes,0.30 ✗,1.00
datasets from NASA,graph,yes,0.15 ✗,1.00
resources tagged climate,graph,yes,0.02 ✗,0.99
papers affiliated with USGS,graph,yes,0.10 ✗,0.99


## Your own inputs

Edit the lists and run the cell. Both models are reloaded, which takes about 10–15 s each on CPU.

In [8]:
MY_QUERIES = ["trending flood datasets from NOAA for Louisiana", "notebooks about kriging"]
MY_REQUESTS = [("Make a 500 m buffer around each school.", "schools.geojson (points; columns: name)"),
               ("Which county has the most wells?", "wells.csv (points; columns: id, lat, lon); counties.geojson (polygons; columns: GEOID, NAME)")]

out = []
for name, path in CHECKPOINTS.items():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = Agent(str(path), device=DEVICE)
    for q in MY_QUERIES:
        a = model.system_one(q, ROUTER_WORDING[name])["answers"]
        out.append({"input": q, "model": name,
                    "answer": f"P(graph) {a['graph']['noul']:.2f} · P(spatial) {a['spatial']['noul']:.2f}"})
    for req, files in MY_REQUESTS:
        p = model.system_one(f"Attached files: {files}\nRequest: {req}", TOOL_Q,
                             max_len=1600, head_max_len=1400)["answers"]["tool"]["probabilities"]
        out.append({"input": req, "model": name,
                    "answer": " · ".join(f"{k} {v:.2f}" for k, v in sorted(p.items(), key=lambda kv: -kv[1])[:3])})
    del model
pd.DataFrame(out).set_index(["input", "model"])

,,answer
input,model,
trending flood datasets from NOAA for Louisiana,vanilla,P(graph) 0.06 · P(spatial) 0.12
notebooks about kriging,vanilla,P(graph) 0.01 · P(spatial) 0.07
Make a 500 m buffer around each school.,vanilla,add_map_layer 0.67 · request_capability 0.09 · count_points_in_are...
Which county has the most wells?,vanilla,add_map_layer 0.49 · cluster_points 0.13 · count_points_in_areas 0.08
trending flood datasets from NOAA for Louisiana,fine-tuned,P(graph) 0.99 · P(spatial) 1.00
notebooks about kriging,fine-tuned,P(graph) 0.00 · P(spatial) 0.00
Make a 500 m buffer around each school.,fine-tuned,buffer_layer 1.00 · execute_code 0.00 · admin_boundary 0.00
Which county has the most wells?,fine-tuned,admin_boundary 0.29 · select_by_attribute 0.09 · vector_spatial_jo...
